# LIMPIEZA DE DATOS

## PASO 1:  Importacion y carga Inicial

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('dataset proceso residentado medico - CONSOLIDADO.csv')
display(df.head(3))

,Año,No,CMP,No Doc.,Apellido Paterno,Apellido Materno,Nombres,Universidad,Tipo,Especialidad/SubEspecialidad,...,ENAM,Examen,Not. Final,Obs.,Ingresó si/no,Universidad (INGRESO),Tipo (INGRESO),Especialidad/SubEspecialidad (INGRESO),Modalidad (INGRESO),Sede (INGRESO)
0,2023,1,93552,73815684,YAURI,HUAMAN,MARTIN EFREN,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,1.5 pts.,60.4,73.870,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
1,2023,2,88328,71701411,CARBAJAL,DIAZ,MARIANA DEL CARMEN,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,2 pts.,57.2,71.173,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
2,2023,3,95865,46024669,BUSTAMANTE,CARDENAS,EDGAR MIGUEL,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,1 pts.,56.0,69.893,NaN,SI,URP,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins


## Paso 2 : Exploracion y estandarizacion de columnas

#### Antes de limpiar los valores internos, es una buena práctica estandarizar los nombres de las columnas para evitar errores tipográficos al codificar.

In [7]:
df.info()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('.', '')
print("\nNuevas columnas:", df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 22329 entries, 0 to 22328
Data columns (total 26 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Año                                     22329 non-null  int64  
 1   No                                      22329 non-null  int64  
 2   CMP                                     22329 non-null  str    
 3   No Doc.                                 22329 non-null  int64  
 4     Apellido Paterno                      22328 non-null  str    
 5   Apellido Materno                        22323 non-null  str    
 6   Nombres                                 22329 non-null  str    
 7   Universidad                             22329 non-null  str    
 8   Tipo                                    22329 non-null  str    
 9   Especialidad/SubEspecialidad            22329 non-null  str    
 10  Modalidad                               22329 non-null  str    
 11  

## Paso 3: Tratamiento de duplicados

### Para asegurar la exactitud de los datos, eliminaremos cualquier registro que se haya duplicado en la consolidación del archivo.

In [8]:
duplicados_previos = df.duplicated().sum()
print(f"Filas duplicadas encontradas: {duplicados_previos}")

df = df.drop_duplicates()

Filas duplicadas encontradas: 0


## Paso 4 : Limpieza profunda y trasnformacion de tipos de datos

### Se extraeran solo los numeros y no las letras y se colocara cero a las columnas vacias de bonificacion porque no obtubieron puntos extra

In [10]:
# 1. Limpieza de las columnas de bonificación (esta parte se  rellena con 0)
columnas_bonificacion = ['serum', '1er_niv', '5to_sup']

def limpiar_puntajes_bonos(valor):
    if pd.isna(valor):
        return 0.0 # Rellenamos con 0 a quienes no tienen bono
    valor_str = str(valor).replace('pts.', '').replace('pts', '').replace(',', '').strip()
    try:
        return float(valor_str)
    except ValueError:
        return 0.0

for col in columnas_bonificacion:
    if col in df.columns: 
        df[col] = df[col].apply(limpiar_puntajes_bonos)


# 2. Limpieza de notas principales quitando el texto, pero MANTENIENDO los nulos (NaN)
columnas_notas = ['prom_pre', 'enam', 'examen']

for col in columnas_notas:
    if col in df.columns:
        # Forzamos a que sea texto, reemplazamos 'pts.' y limpiamos espacios vacíos
        df[col] = df[col].astype(str).str.replace('pts.', '', regex=False)\
                                     .str.replace('pts', '', regex=False)\
                                     .str.strip()
        
        # pd.to_numeric con errors='coerce' convierte el texto limpio a decimal (float). 
        # Si encuentra un vacío ("nan"), lo deja matemáticamente nulo (NaN) en lugar de poner cero.
        df[col] = pd.to_numeric(df[col], errors='coerce')


# 3. Ahora sí, de manera segura, eliminamos los registros que no tienen notas
df.dropna(subset=['prom_pre', 'enam', 'examen'], inplace=True)

df.head(3)

,año,no,cmp,no_doc,apellido_paterno,apellido_materno,nombres,universidad,tipo,especialidad/subespecialidad,...,enam,examen,not_final,obs,ingresó_si/no,universidad_(ingreso),tipo_(ingreso),especialidad/subespecialidad_(ingreso),modalidad_(ingreso),sede_(ingreso)
0,2023,1,93552,73815684,YAURI,HUAMAN,MARTIN EFREN,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,1.5,60.4,73.870,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
1,2023,2,88328,71701411,CARBAJAL,DIAZ,MARIANA DEL CARMEN,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,2.0,57.2,71.173,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
2,2023,3,95865,46024669,BUSTAMANTE,CARDENAS,EDGAR MIGUEL,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,1.0,56.0,69.893,NaN,SI,URP,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins


## Paso 5 : Eliminar datos privados

### Aqui eliminaremos las columnas de los datos personales 

In [11]:
columnas_personales = ['apellido_paterno', 'apellido_materno', 'nombres','cmp']
df.drop(columns=columnas_personales, inplace=True, errors='ignore')
print("Columnas actuales en el dataset:")
df.head(5)

Columnas actuales en el dataset:


,año,no,no_doc,universidad,tipo,especialidad/subespecialidad,modalidad,serum,sncds,1er_niv,...,enam,examen,not_final,obs,ingresó_si/no,universidad_(ingreso),tipo_(ingreso),especialidad/subespecialidad_(ingreso),modalidad_(ingreso),sede_(ingreso)
0,2023,1,73815684,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,...,1.5,60.4,73.870,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
1,2023,2,71701411,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,...,2.0,57.2,71.173,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
2,2023,3,46024669,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,...,1.0,56.0,69.893,NaN,SI,URP,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
3,2023,4,46374473,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,...,1.5,56.0,69.346,NaN,SI,UNFV,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
4,2023,5,76428325,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,...,1.0,56.0,68.849,NaN,SI,UPCH,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Cayetano Heredia


## Paso 6: Eliminar columnas no necesarias

### Eliminamos las columnas numero de orden y observaciones

In [16]:
columnas_a_eliminar = ["no", "obs"]
df.drop(columns=columnas_a_eliminar, inplace=True, errors="ignore")

print("Se eliminaron las columnas 'no' y 'obs' (si existían).")
df.head(5)

Se eliminaron las columnas 'no' y 'obs' (si existían).


,año,no_doc,universidad,tipo,especialidad/subespecialidad,modalidad,serum,sncds,1er_niv,5to_sup,...,not_final,ingresó_si/no,universidad_(ingreso),tipo_(ingreso),especialidad/subespecialidad_(ingreso),modalidad_(ingreso),sede_(ingreso),total_bonificacion,rango_pregrado,nota_cv
0,2023,73815684,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,0.0,...,73.870,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins,10.0,Bajo,13.470
1,2023,71701411,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,0.0,...,71.173,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins,10.0,Bajo,13.973
2,2023,46024669,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,1.0,...,69.893,SI,URP,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins,11.0,Bajo,13.893
3,2023,46374473,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,0.0,...,69.346,SI,UNFV,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins,10.0,Bajo,13.346
4,2023,76428325,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,NaN,0.0,0.0,...,68.849,SI,UPCH,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Cayetano Heredia,10.0,Bajo,12.849


## Paso 7 : Creación de variables derivadas

### Para analizar la influencia de las bonificaciones en la probabilidad de ingreso, crearemos una nueva columna que sume todos los puntos extra

In [17]:
df['total_bonificacion'] = df['serum'] + df['1er_niv'] + df['5to_sup']

display(df[['serum', '1er_niv', '5to_sup', 'total_bonificacion']].head(5))

,serum,1er_niv,5to_sup,total_bonificacion
0,10.0,0.0,0.0,10.0
1,10.0,0.0,0.0,10.0
2,10.0,0.0,1.0,11.0
3,10.0,0.0,0.0,10.0
4,10.0,0.0,0.0,10.0


## Paso 8 : Crear variable de ponderacion

### Representaría la ponderación del currículum vitae u otros factores evaluativos aparte del examen

In [18]:
# 1. Asegurarnos de que 'not_final' esté limpia y sea numérica (por si tiene el texto "pts")
df['not_final'] = df['not_final'].astype(str).str.replace('pts.', '', regex=False)\
                                 .str.replace('pts', '', regex=False)\
                                 .str.strip()

# Convertir a valor numérico decimal (float)
df['not_final'] = pd.to_numeric(df['not_final'], errors='coerce')

# 2. Crear la nueva columna 'nota_cv' (Nota Final - Examen)
df['nota_cv'] = df['not_final'] - df['examen']

# 3. Mostrar las primeras filas de estas 3 columnas para verificar que la resta se hizo bien
print("Verificación de la nueva variable 'nota_cv':")
display(df[['examen', 'not_final', 'nota_cv']].head())

Verificación de la nueva variable 'nota_cv':


,examen,not_final,nota_cv
0,60.4,73.870,13.470
1,57.2,71.173,13.973
2,56.0,69.893,13.893
3,56.0,69.346,13.346
4,56.0,68.849,12.849


## Paso 9: Exportar a dataset limpio

### En este paso guardamos el DataFrame ya depurado en un archivo nuevo para conservar los datos limpios y no sobrescribir el archivo original

In [19]:
df.to_csv('dataset_limpio.csv', index=False, encoding='utf-8-sig')
print('Nuevo dataset creado: dataset_limpio.csv')

Nuevo dataset creado: dataset_limpio.csv
